# Brain Tumor — Kaggle GPU training

Runs the whole training pipeline on a Kaggle GPU. Everything before this point
(preprocessing, mask generation) is cheap on Kaggle's local disk and painfully
slow on a OneDrive-synced folder, so it all happens here.

**Before running, in the notebook sidebar:**

| Setting | Value |
|---|---|
| Accelerator | GPU T4 x2 (or P100) |
| Internet | On — needed for `git clone`, `pip install`, ImageNet weights |
| Persistence | Variables and files off (outputs are saved explicitly at the end) |

Budget: roughly 20 min detection + 40 min classification + 60 min segmentation,
well inside one 12 h session. Kaggle allows 30 GPU-hours per week.

You can run stages 1-2 and 3 in separate sessions; stage 3 only needs the masks,
not the TensorFlow models.

## 0. Environment check

Fail fast if the accelerator is off — otherwise this silently runs for hours on CPU.

In [ ]:
import tensorflow as tf, torch, keras
print('TF', tf.__version__, '| Keras', keras.__version__, '| Torch', torch.__version__)
gpus = tf.config.list_physical_devices('GPU')
print('TF GPUs:', gpus)
print('Torch CUDA:', torch.cuda.is_available(), torch.cuda.get_device_name(0) if torch.cuda.is_available() else '')
assert gpus, 'No GPU. Set Accelerator to GPU in the sidebar before running.'

## 1. Get the code and data

`Dataset/` is committed to the repo, so the clone brings the 7200 images with it
(~150 MB). Nothing to upload to Kaggle separately.

In [ ]:
REPO_URL = 'https://github.com/Mounika-Reddy-0802/Brain-Tumor-Detection-and-Segmentation-.git'
REPO_DIR = 'Brain-Tumor-Detection-and-Segmentation-'

import os, pathlib
os.chdir('/kaggle/working')
if not pathlib.Path(REPO_DIR).exists():
    !git clone --depth 1 $REPO_URL
os.chdir(f'/kaggle/working/{REPO_DIR}')
print('cwd:', os.getcwd())
!ls Dataset/Training && du -sh Dataset

In [ ]:
# smp is not preinstalled on Kaggle; everything else already is.
!pip install -q segmentation-models-pytorch
import segmentation_models_pytorch as smp; print('smp', smp.__version__)

## 2. Prepare data (~3 min)

Writes cropped + CLAHE-enhanced 224x224 images to `data/processed`, then the Otsu
pseudo-masks to `data/masks`. Doing this once means the training loops only decode
JPEGs instead of re-running OpenCV every epoch.

In [ ]:
!python -m src.data.preprocessing --raw-dir Dataset --out-dir data/processed --img-size 224
!python -m scripts.generate_masks --config configs/config_processed.yaml
!find data/processed -name '*.jpg' | wc -l && find data/masks -name '*.png' | wc -l

## 3. Detection — binary tumor / no-tumor (~20 min)

5 warmup epochs with the backbone frozen, then up to 32 fine-tuning epochs with
early stopping on `val_recall` (missed tumors matter more than false alarms).

In [ ]:
!python -m src.training.train_tf_detection --config configs/config_processed.yaml

## 4. Classification — 4-class (~40 min)

In [ ]:
!python -m src.training.train_tf_classification --config configs/config_processed.yaml

## 5. Segmentation — U-Net (~60 min)

Trains against the precomputed weak labels: the brightest compact blob inside the
brain, with `notumor` scans given an empty mask. Roughly a third of those masks
land on the actual lesion; the rest catch eye globes, skull-base structures or
ventricles. Treat the resulting Dice as agreement with that heuristic, not as
tumour localisation accuracy.

For genuine segmentation quality this needs expert annotations — see the
segmentation note in the README.

In [ ]:
!python -m src.training.train_torch_segmentation --config configs/config_processed.yaml

## 6. Evaluate on the held-out Testing split

These are the numbers to quote in the report — the values printed during training are validation-split numbers.

In [ ]:
!python -m src.evaluation.evaluate_all --config configs/config_processed.yaml
import json; print(json.dumps(json.load(open('outputs/evaluation_report.json')), indent=2))

## 7. Save the artifacts

`/kaggle/working` is wiped when the session ends. Commit the notebook (Save Version →
Save & Run All) so the output files persist, then download `trained_models.zip`.

In [ ]:
import shutil, os
os.makedirs('/kaggle/working/artifacts', exist_ok=True)
for src in ['models', 'checkpoints', 'outputs', 'logs']:
    if os.path.exists(src):
        shutil.copytree(src, f'/kaggle/working/artifacts/{src}', dirs_exist_ok=True)
shutil.make_archive('/kaggle/working/trained_models', 'zip', '/kaggle/working/artifacts')
print(os.path.getsize('/kaggle/working/trained_models.zip') / 1e6, 'MB')
!ls -la /kaggle/working/artifacts/models